# Bulk FHIR Data Analysis

This notebook loads Bulk FHIR NDJSON files and produces:
1. Patient demographics summary
2. Top 10 diagnosis codes
3. Medication frequency analysis


## Expected Input Files

Place these files in the same folder as this notebook:

- `Patient.ndjson`
- `Condition.ndjson`
- `MedicationRequest.ndjson`


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
def load_ndjson(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


## Load Bulk FHIR NDJSON Files


In [ ]:
patients = load_ndjson("Patient.ndjson")
conditions = load_ndjson("Condition.ndjson")
medications = load_ndjson("MedicationRequest.ndjson")

print("Patients:", len(patients))
print("Conditions:", len(conditions))
print("MedicationRequests:", len(medications))


## 1. Patient Demographics Summary


In [ ]:
patient_rows = []
for p in patients:
    patient_rows.append({
        "id": p.get("id"),
        "gender": p.get("gender"),
        "birthDate": p.get("birthDate")
    })

patient_df = pd.DataFrame(patient_rows)
print("Patient Demographics Summary")
display(patient_df)
patient_df.describe(include="all")


## 2. Top 10 Diagnosis Codes / Conditions


In [ ]:
condition_rows = []
for c in conditions:
    code_text = c.get("code", {}).get("text")
    if not code_text:
        codings = c.get("code", {}).get("coding", [])
        code_text = codings[0].get("display") if codings else None

    condition_rows.append({
        "condition": code_text
    })

condition_df = pd.DataFrame(condition_rows)
top_conditions = condition_df["condition"].value_counts().head(10)
print("Top 10 Diagnosis Codes / Conditions")
print(top_conditions)

top_conditions.plot(kind="bar", title="Top 10 Diagnosis Codes / Conditions")
plt.xlabel("Diagnosis / Condition")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 3. Medication Frequency Analysis


In [ ]:
med_rows = []
for m in medications:
    med_text = m.get("medicationCodeableConcept", {}).get("text")
    if not med_text:
        codings = m.get("medicationCodeableConcept", {}).get("coding", [])
        med_text = codings[0].get("display") if codings else None

    med_rows.append({
        "medication": med_text
    })

med_df = pd.DataFrame(med_rows)
med_freq = med_df["medication"].value_counts()
print("Medication Frequency")
print(med_freq)

med_freq.plot(kind="bar", title="Medication Frequency")
plt.xlabel("Medication")
plt.ylabel("Count")
plt.tight_layout()
plt.show()
